# Classification de cellules sanguines -- 12 classes -- sortie à 3 états

**Objectif** : à partir d'une image de cellule 400x400 px, prédire son type parmi 12 classes cibles,
avec une sortie à 3 états :
- **Test bon** : confiance suffisante pour garantir ~99,9% de précision (calibré sur un jeu de validation)
- **Résultat indéterminé** : confiance dans une zone grise, prédiction non garantie
- **Test pas bon** : confiance trop faible, le modèle n'est pas fiable sur cette image

**Deux précisions par rapport à la demande initiale :**
1. *"Table de convolution"* → je fournis une **matrice de confusion** (la métrique standard pour évaluer
   un classifieur multi-classes). La convolution est une opération interne au réseau (CNN), pas une métrique de performance.
2. *RMSE pour de la classification* → implémentée comme **score de Brier** : RMSE entre le vecteur
   one-hot de la vraie classe et le vecteur de probabilités softmax prédit. C'est à la fois la fonction
   de perte utilisée à l'entraînement et la métrique de suivi.

**Classes cibles (12)** : Basophil, Eosinophil, Erythroblast, Lymphocyte, Metamyelocyte, Monocyte,
Myeloblast, Myelocyte, Neutrophil (band), Neutrophil (segmented), Platelet, Promyelocyte.

**Prérequis Kaggle** : activer un accélérateur GPU (Settings > Accelerator > GPU T4/P100) -- l'entraînement
sur CPU serait beaucoup trop lent pour un CNN sur images 400x400.

## 1. Configuration

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chemins en Path dès le début
LMU_DIR = Path("/kaggle/input/datasets/jfonlupt/pkg-aml-cytomorphology-lmu/PKG-AML-Cytomorphology_LMU")
PBC_DIR = Path("/kaggle/input/datasets/jfonlupt/pbc-dataset-normal-dib/PBC_dataset_normal_DIB/PBC_dataset_normal_DIB")
ANNOTATIONS_FILE = LMU_DIR / "annotations.dat"

# Nombre d'images échantillonnées par classe pour le calcul des indicateurs
# (mettre None pour traiter TOUTES les images -- plus lent selon la taille du dataset)
SAMPLE_PER_CLASS = 300
RANDOM_SEED = 42

print("LMU_DIR:", LMU_DIR)
print("ANNOTATIONS_FILE:", ANNOTATIONS_FILE)
print("LMU_DIR existe ?", LMU_DIR.exists())
print("PBC_DIR existe ?", PBC_DIR.exists())
print("ANNOTATIONS_FILE existe ?", ANNOTATIONS_FILE.exists())

# --- Paramètres spécifiques à la classification ---
IMG_SIZE = 400                 # dimension cible (400x400 comme demandé)
BATCH_SIZE = 16
EPOCHS = 30
TARGET_PRECISION = 0.999       # précision visée sur le bucket "Test bon"
MIN_SUPPORT_CALIBRATION = 30   # nb minimal d'échantillons pour valider un seuil
REJECT_FLOOR = 0.5             # en dessous de cette précision cumulée -> "Test pas bon"

np.random.seed(RANDOM_SEED)


## 2. Classes cibles et correspondance avec les codes des 2 datasets

Les deux datasets utilisent des nomenclatures différentes et plus fines que les 12 classes demandées.
On mappe explicitement chaque code source vers une des 12 classes cibles, et on **exclut volontairement**
les codes qui ne correspondent à aucune des 12 classes (pour ne pas introduire de bruit d'étiquetage) :
- `KSC` (cellule "smudge", artefact d'écrasement -- pas un vrai type cellulaire)
- `MOB` (monoblaste -- absent de la liste demandée)
- `IG` côté PBC (granulocyte immature générique, catégorie fourre-tout ambiguë entre myélocyte /
  métamyélocyte / promyélocyte / bande -- les codes plus précis `MY`, `MMY`, `PMY`, `BNE` sont
  utilisés à la place quand ils existent)

In [ ]:
TARGET_CLASSES = [
    "Neutrophil (segmented)", "Lymphocyte", "Eosinophil", "Myeloblast",
    "Monocyte", "Platelet", "Neutrophil (band)", "Erythroblast",
    "Basophil", "Myelocyte", "Metamyelocyte", "Promyelocyte",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
N_CLASSES = len(TARGET_CLASSES)

LMU_TO_TARGET = {
    "NGS": "Neutrophil (segmented)",
    "NGB": "Neutrophil (band)",
    "LYT": "Lymphocyte",
    "LYA": "Lymphocyte",
    "EOS": "Eosinophil",
    "MYO": "Myeloblast",
    "MON": "Monocyte",
    "EBO": "Erythroblast",
    "BAS": "Basophil",
    "MYB": "Myelocyte",
    "MMZ": "Metamyelocyte",
    "PMO": "Promyelocyte",
    "PMB": "Promyelocyte",
    # KSC, MOB : volontairement absents (voir markdown ci-dessus)
}

PBC_TO_TARGET = {
    "BA": "Basophil",
    "EO": "Eosinophil",
    "ERB": "Erythroblast",
    "LY": "Lymphocyte",
    "MO": "Monocyte",
    "BNE": "Neutrophil (band)",
    "SNE": "Neutrophil (segmented)",
    "NEUTROPHIL": "Neutrophil (segmented)",
    "PMY": "Promyelocyte",
    "MY": "Myelocyte",
    "MMY": "Metamyelocyte",
    "PLATELET": "Platelet",
    # IG : volontairement absent (voir markdown ci-dessus)
}

# Vérification : chaque classe cible doit avoir au moins une source de données
missing = set(TARGET_CLASSES) - set(LMU_TO_TARGET.values()) - set(PBC_TO_TARGET.values())
assert not missing, f"Classes cibles sans aucune source de données : {missing}"
print("Mapping validé -- toutes les classes cibles ont au moins une source de données.")


## 3. Construction du dataset unifié (12 classes)

In [ ]:
def extract_pbc_subclass(filename_stem):
    """Extrait le code de sous-classe PBC à partir du préfixe du nom de fichier
    (ex: 'BA_47' -> 'BA', 'PLATELET_516' -> 'PLATELET')."""
    return filename_stem.split("_")[0].upper()


def load_LMU_labeled():
    df = pd.read_csv(
        ANNOTATIONS_FILE, sep=" ", header=None,
        names=["relative_path", "class_code", "re_annotation", "re_re_annotation"]
    )
    df["path"] = df["relative_path"].apply(lambda p: LMU_DIR / p)
    df["exists"] = df["path"].apply(lambda p: p.exists())

    def final_code(row):
        for v in (row["re_re_annotation"], row["re_annotation"]):
            if pd.notna(v) and str(v) != "nan":
                return v
        return row["class_code"]

    df["source_code"] = df.apply(final_code, axis=1)
    df["target_label"] = df["source_code"].map(LMU_TO_TARGET)
    df["source_dataset"] = "LMU"
    return df[df["exists"] & df["target_label"].notna()][["path", "target_label", "source_dataset"]]


def load_PBC_labeled():
    if not PBC_DIR.exists():
        raise FileNotFoundError(f"PBC_DIR introuvable : {PBC_DIR}")
    rows = []
    for class_dir in sorted(PBC_DIR.iterdir()):
        if class_dir.is_dir():
            for f in class_dir.iterdir():
                if f.suffix.lower() in (".png", ".jpg", ".jpeg", ".tif", ".tiff"):
                    rows.append({"path": f, "source_code": extract_pbc_subclass(f.stem)})
    if not rows:
        raise ValueError(f"Aucune image trouvée dans {PBC_DIR}")
    df = pd.DataFrame(rows)
    df["target_label"] = df["source_code"].map(PBC_TO_TARGET)
    df["source_dataset"] = "PBC"
    return df[df["target_label"].notna()][["path", "target_label", "source_dataset"]]


lmu_labeled = load_LMU_labeled()
pbc_labeled = load_PBC_labeled()
data = pd.concat([lmu_labeled, pbc_labeled], ignore_index=True)
data["label_idx"] = data["target_label"].map(CLASS_TO_IDX)

print(f"Total images exploitables : {len(data)} (LMU: {len(lmu_labeled)} | PBC: {len(pbc_labeled)})")
print()
print(data.groupby(["target_label", "source_dataset"]).size().unstack(fill_value=0))


## 4. Split stratifié train / validation / test

Le split validation sert à **calibrer les 2 seuils de confiance** (section 8), le split test sert uniquement à **évaluer** le modèle final -- il n'est jamais utilisé pour ajuster quoi que ce soit, afin d'obtenir une estimation non biaisée de la précision réelle.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data, test_size=0.30, stratify=data["target_label"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["target_label"], random_state=RANDOM_SEED
)

print(f"Train : {len(train_df)} | Validation : {len(val_df)} | Test : {len(test_df)}")
train_df["target_label"].value_counts()


## 5. Pipeline de données (tf.data) et pondération des classes

Les classes sont déséquilibrées (ex. `Myeloblast` uniquement issue de LMU, `Platelet` uniquement de PBC). On calcule des poids de classe pour compenser lors de l'entraînement.

In [ ]:
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])  # redimensionne toute image vers 400x400
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


def make_dataset(df, training=False):
    paths = df["path"].astype(str).values
    labels = tf.one_hot(df["label_idx"].values, N_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=RANDOM_SEED)
        augment = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal_and_vertical"),
            tf.keras.layers.RandomRotation(0.05),
            tf.keras.layers.RandomBrightness(0.05),
            tf.keras.layers.RandomContrast(0.05),
        ])
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(N_CLASSES),
    y=train_df["label_idx"].values,
)
class_weight = {i: w for i, w in enumerate(class_weights_arr)}
print("Poids de classe :", {IDX_TO_CLASS[i]: round(w, 2) for i, w in class_weight.items()})


## 6. Modèle : transfer learning + perte RMSE (score de Brier)

Backbone `EfficientNetB0` pré-entraîné sur ImageNet, tête de classification personnalisée.
**La fonction de perte est le score de Brier** (= MSE entre le softmax prédit et le one-hot de la vraie
classe), ce qui correspond directement à la demande d'optimiser sur une métrique de type RMSE
(`RMSE = sqrt(MSE)`). On garde `accuracy` comme métrique de suivi secondaire, plus intuitive à lire.

In [ ]:
def rmse_metric(y_true, y_pred):
    """RMSE multiclasse (racine du score de Brier) entre le one-hot de la vraie classe
    et le vecteur de probabilités prédit -- métrique demandée pour évaluer le modèle."""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))


def build_model():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # phase 1 : on entraîne uniquement la tête

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs * 255.0)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(N_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.MeanSquaredError(),   # <- score de Brier = perte RMSE au carré
        metrics=[rmse_metric, "accuracy"],
    )
    return model, base


model, base_model = build_model()
model.summary()


## 7. Entraînement

Deux phases : (1) tête seule, backbone gelé ; (2) fine-tuning des dernières couches du backbone à taux d'apprentissage réduit.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_rmse_metric", patience=6, restore_best_weights=True, mode="min"),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_rmse_metric", factor=0.5, patience=3, mode="min"),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_rmse_metric", save_best_only=True, mode="min"),
]

# Phase 1 : entraînement de la tête uniquement
history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)

# Phase 2 : fine-tuning des 30 dernières couches du backbone
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[rmse_metric, "accuracy"],
)
history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for h, label in [(history_head, "tête"), (history_finetune, "fine-tuning")]:
    axes[0].plot(h.history["rmse_metric"], label=f"train ({label})")
    axes[0].plot(h.history["val_rmse_metric"], label=f"val ({label})", linestyle="--")
    axes[1].plot(h.history["accuracy"], label=f"train ({label})")
    axes[1].plot(h.history["val_accuracy"], label=f"val ({label})", linestyle="--")
axes[0].set_title("RMSE (score de Brier)"); axes[0].set_xlabel("Époque"); axes[0].legend(fontsize=7)
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Époque"); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()


## 8. Calibration des seuils de confiance (sortie à 3 états)

On calcule les probabilités prédites sur le jeu de **validation**, puis on détermine :
- `t_high` : confiance minimale garantissant une précision >= 99,9% (**Test bon**)
- `t_low` : en dessous, la précision cumulée retombe sous 50% (**Test pas bon**)
- entre les deux : **Résultat indéterminé**

Ces seuils sont calibrés uniquement sur la validation, puis appliqués tels quels sur le test (section 9).

In [ ]:
def calibrate_thresholds(y_true, y_pred, confidences, target_precision=TARGET_PRECISION,
                          reject_floor=REJECT_FLOOR, min_support=MIN_SUPPORT_CALIBRATION):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); confidences = np.asarray(confidences)
    order = np.argsort(-confidences)
    conf_sorted = confidences[order]
    correct_sorted = (y_true[order] == y_pred[order]).astype(int)

    cum_precision = np.cumsum(correct_sorted) / np.arange(1, len(correct_sorted) + 1)
    cum_count = np.arange(1, len(correct_sorted) + 1)

    def last_threshold_meeting(floor):
        for i in range(len(cum_precision) - 1, -1, -1):
            if cum_count[i] >= min_support and cum_precision[i] >= floor:
                return conf_sorted[i]
        return None

    t_high = last_threshold_meeting(target_precision)
    t_low = last_threshold_meeting(reject_floor)
    if t_high is None:
        t_high = 1.01  # la précision cible n'est atteinte à aucun seuil : bucket "Test bon" vide
    if t_low is None:
        t_low = float(conf_sorted.min())
    t_low = min(t_low, t_high)
    return float(t_high), float(t_low)


def classify_status(confidence, t_low, t_high):
    if confidence >= t_high:
        return "Test bon"
    elif confidence < t_low:
        return "Test pas bon"
    return "Résultat indéterminé"


val_proba = model.predict(val_ds, verbose=0)
val_pred_idx = val_proba.argmax(axis=1)
val_confidence = val_proba.max(axis=1)
val_true_idx = val_df["label_idx"].values

T_HIGH, T_LOW = calibrate_thresholds(val_true_idx, val_pred_idx, val_confidence)
print(f"Seuil 'Test bon'    (t_high) = {T_HIGH:.4f}")
print(f"Seuil 'Test pas bon' (t_low) = {T_LOW:.4f}")


## 9. Évaluation finale sur le jeu de test

Le jeu de test n'a servi ni à l'entraînement, ni à la calibration des seuils : c'est la mesure la plus fiable de performance réelle.

In [ ]:
def brier_rmse(y_true_idx, proba_matrix, n_classes):
    onehot = np.eye(n_classes)[y_true_idx]
    return float(np.sqrt(np.mean((onehot - proba_matrix) ** 2)))


test_proba = model.predict(test_ds, verbose=0)
test_pred_idx = test_proba.argmax(axis=1)
test_confidence = test_proba.max(axis=1)
test_true_idx = test_df["label_idx"].values

test_rmse = brier_rmse(test_true_idx, test_proba, N_CLASSES)
test_accuracy = (test_pred_idx == test_true_idx).mean()
print(f"RMSE (score de Brier) sur le test : {test_rmse:.4f}")
print(f"Accuracy globale sur le test      : {test_accuracy*100:.2f}%")

statuses = np.array([classify_status(c, T_LOW, T_HIGH) for c in test_confidence])
print()
print(f"{'Statut':22s} {'n':>6s} {'% du test':>10s} {'précision':>10s}")
for s in ["Test bon", "Résultat indéterminé", "Test pas bon"]:
    mask = statuses == s
    n_s = mask.sum()
    acc_s = (test_true_idx[mask] == test_pred_idx[mask]).mean() if n_s > 0 else float("nan")
    print(f"{s:22s} {n_s:6d} {n_s/len(test_df)*100:9.1f}% {acc_s*100:9.2f}%")


## 10. Matrice de confusion

Deux matrices : une sur l'ensemble du jeu de test (toutes prédictions confondues), une restreinte au bucket **Test bon** (pour vérifier visuellement que les 99,9% de précision visés s'y concentrent bien).

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title, normalize=True):
    n = len(class_names)
    cm = np.zeros((n, n), dtype=float)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        cm_display = cm / row_sums
        fmt = "{:.0%}"
    else:
        cm_display = cm
        fmt = "{:.0f}"

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm_display, cmap="Blues", vmin=0, vmax=max(cm_display.max(), 1e-6))
    ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticks(range(n)); ax.set_yticklabels(class_names)
    ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle")
    ax.set_title(title)
    for i in range(n):
        for j in range(n):
            if cm[i, j] > 0:
                ax.text(j, i, fmt.format(cm_display[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if cm_display[i, j] > cm_display.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(test_true_idx, test_pred_idx, TARGET_CLASSES,
                       "Matrice de confusion -- ensemble du test")

bon_mask = statuses == "Test bon"
if bon_mask.sum() > 0:
    plot_confusion_matrix(test_true_idx[bon_mask], test_pred_idx[bon_mask], TARGET_CLASSES,
                           "Matrice de confusion -- bucket 'Test bon' uniquement")


## 11. Fonction d'inférence sur une nouvelle image

In [ ]:
def predict_cell(image_path, model=model, t_low=T_LOW, t_high=T_HIGH):
    """Prédit le type d'une cellule à partir d'une image, avec sortie à 3 états."""
    img, _ = load_and_preprocess(str(image_path), tf.zeros(N_CLASSES))
    proba = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    confidence = float(np.max(proba))
    status = classify_status(confidence, t_low, t_high)
    return {
        "classe_predite": IDX_TO_CLASS[pred_idx],
        "confiance": round(confidence, 4),
        "statut": status,
        "top3": sorted(
            [(IDX_TO_CLASS[i], round(float(p), 4)) for i, p in enumerate(proba)],
            key=lambda x: -x[1]
        )[:3],
    }


# Exemple d'utilisation :
# result = predict_cell(test_df["path"].iloc[0])
# print(result)


## Conclusion

- Le modèle est entraîné avec une **perte RMSE** (score de Brier) sur les probabilités prédites, comme demandé.
- La sortie à **3 états** (`Test bon` / `Résultat indéterminé` / `Test pas bon`) repose sur deux seuils de
  confiance calibrés sur un jeu de **validation**, jamais sur le test, pour éviter tout biais optimiste.
- La cible de **99,9% de précision** s'applique uniquement au bucket `Test bon` -- c'est un choix
  fondamental de la classification sélective (*selective classification*) : on accepte de ne pas trancher
  sur les cas ambigus (`indéterminé`/`pas bon`) plutôt que de se tromper sur des cas qu'on annonce comme fiables.
- La **matrice de confusion** (et non "table de convolution") permet de visualiser les confusions résiduelles,
  notamment entre classes morphologiquement proches (ex. `Neutrophil (band)` vs `Neutrophil (segmented)`,
  ou `Myelocyte` vs `Metamyelocyte` vs `Promyelocyte`, qui forment un continuum de maturation).

**Pistes d'amélioration si la précision de 99,9% est difficile à atteindre en pratique :**
- Augmenter `MIN_SUPPORT_CALIBRATION` et la taille du jeu de validation (la précision empirique sur peu
  d'échantillons est peu fiable statistiquement)
- Utiliser une calibration plus rigoureuse (température de calibration / isotonic regression) avant
  de fixer les seuils
- Envisager un modèle plus grand ou plus de données pour les classes les plus confondues (cf. matrice
  de confusion), en particulier `Myeloblast` qui ne provient que d'une seule source (LMU).